# Week 15 — Transformers: Everything Looks at Everything

**Course:** Applied ML Foundations for SaaS Analytics  
**Who this is for:** Engineers who have written a search index, a join, or `dict.get`. This is the architecture behind GPT, BERT, Copilot, and most of LangChain.

---

## 🎯 What you will be able to do

- Explain **attention** as a soft dictionary lookup: query → keys → weighted values
- See why we add **position** (the model has no loop, so it cannot “know” order otherwise)
- Build a tiny self-attention block in PyTorch and watch weights light up
- Classify CloudWave **feedback text** with a small Transformer encoder
- Know encoder vs decoder vs “the API you will actually call”

<div class="think-box">
<strong>Think of it like… a database lookup where every row is a candidate, and the score is “how related are you to my question?”</strong>
<p><strong>Query (Q)</strong> = what this token is looking for.<br>
<strong>Keys (K)</strong> = what every token advertises it contains.<br>
<strong>Values (V)</strong> = the payload you actually mix in if the key matched.</p>
<p>Attention weights are a softmax over “how well does my query match each key.” Then you take the weighted sum of values. No clipboard. No left-to-right bottleneck. Every token does this <em>in parallel</em>.</p>
</div>



<div class="cue-box">
<strong>Laptop budget</strong>
<p>No GPU. Aimed at ~8&nbsp;GB RAM. Training uses a few thousand sampled customers (or short sequences) so this notebook should finish in a <strong>few minutes on CPU</strong>. The ideas are the same if you later set <code>n=None</code> and train on all 50k rows.</p>
</div>


In [ ]:
%matplotlib inline
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the shared style kit importable whether you launch from notebooks/ or repo root
for _p in [Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent]:
    if (_p / "course_style.py").exists():
        sys.path.insert(0, str(_p))
        break

from course_style import apply_style, setup_plots, find_data_dir

apply_style()
setup_plots()
DATA = find_data_dir()
print(f"Data folder: {DATA}")
print("Laptop mode: no GPU required. Models use a sample so each week finishes in a few minutes on CPU.")


try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as exc:
    raise SystemExit("PyTorch is missing. Install with:  pip install torch") from exc

torch.manual_seed(0)
DEVICE = torch.device("cpu")
print("torch", torch.__version__, "device", DEVICE)


## The picture

```
tokens:     [  "login" , "failed" , "again" ]
               Q K V      Q K V      Q K V
                 \         |         /
                  \        |        /
                   softmax(Q · Kᵀ)   ← who should I read?
                         │
                    mix of V's
```

<div class="math-box">
<strong>Math, translated</strong>
<p><code>weights = softmax(Q @ K.T / sqrt(d))</code> → a row of positive numbers that sum to 1, one row per token. Divide by <code>sqrt(d)</code> so the dot products do not explode when the vectors are long. Then <code>output = weights @ V</code>. That is attention. Multi-head = several of these lookups in parallel, then concatenated — several reviewers reading for different things.</p>
</div>


In [ ]:
# Tiny self-attention you can print
torch.manual_seed(0)
tokens = ["login", "failed", "again"]
d = 4
X = torch.randn(len(tokens), d)          # pretend embeddings
Wq = torch.randn(d, d); Wk = torch.randn(d, d); Wv = torch.randn(d, d)
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / d ** 0.5
weights = torch.softmax(scores, dim=-1)
out = weights @ V

fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(weights.detach().numpy(), cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(3), tokens); ax.set_yticks(range(3), tokens)
ax.set_xlabel("looking at"); ax.set_ylabel("token")
ax.set_title("Attention weights (rows sum to 1)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()
print("Each row is a probability distribution over who to read.")


## Why position encodings exist

A Transformer is a bag of lookups. `"the movie was not good"` and `"the movie was good not"` look the same unless you **stamp** each token with where it sat.

<div class="engineer-box">
<strong>Engineer mental model</strong>
<p>Position = an extra feature, like adding <code>index</code> to a log line before you embed it. Modern models use rotary / learned positions. You do not need the sine formula. You need: <em>without a position stamp, order is invisible.</em></p>
</div>


## CloudWave: classify feedback text

We will bag-of-words the comments into a short token id sequence and run a toy encoder. This is **not** BERT. It is the moving parts, small enough to train on a laptop in a minute.


In [ ]:
feedback = pd.read_json(DATA / "feedback.json", lines=True)
# Binary: praise vs everything else (or bug vs not)
feedback["y"] = (feedback["category"].str.lower() == "praise").astype(int)
print(feedback["category"].value_counts().head())
print("praise rate", feedback["y"].mean().round(3))

# Character-level tokens — ugly, honest, no extra downloads
def encode(text: str, n=32):
    ids = [min(ord(c), 126) for c in str(text).lower()[:n]]
    ids += [0] * (n - len(ids))
    return ids

# 4k comments is enough to see the loop move — the rest is the same idea
feedback = feedback.sample(n=min(4000, len(feedback)), random_state=0)
ids = np.array([encode(t) for t in feedback["feedback_text"]], dtype=np.int64)
y = feedback["y"].to_numpy(dtype=np.int64)
rng = np.random.default_rng(0)
idx = rng.permutation(len(ids))
cut = int(0.8 * len(ids))
Xtr, Xte = ids[idx[:cut]], ids[idx[cut:]]
ytr, yte = y[idx[:cut]], y[idx[cut:]]
print("seq shape", Xtr.shape, "vocab 0–126")


In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab=127, d=24, nhead=4, ntok=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(ntok, d)
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=nhead,
                                           dim_feedforward=48, batch_first=True,
                                           dropout=0.1)
        self.enc = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(d, 1)

    def forward(self, token_ids):
        b, t = token_ids.shape
        pos = torch.arange(t).unsqueeze(0).expand(b, t)
        x = self.emb(token_ids) + self.pos(pos)
        h = self.enc(x)                       # (B, T, d)
        pooled = h.mean(dim=1)
        return self.head(pooled).squeeze(-1)


model = TinyTransformer()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
print("params", sum(p.numel() for p in model.parameters()))

def batch(X, y, bs=256):
    for i in range(0, len(X), bs):
        yield torch.tensor(X[i:i+bs]), torch.tensor(y[i:i+bs], dtype=torch.float32)

hist = []
for epoch in range(4):
    model.train()
    tr_loss = 0.0
    n = 0
    for xb, yb in batch(Xtr, ytr):
        loss = F.binary_cross_entropy_with_logits(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tr_loss += float(loss) * len(xb); n += len(xb)
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(Xte))
        te_loss = float(F.binary_cross_entropy_with_logits(
            logits, torch.tensor(yte, dtype=torch.float32)))
        acc = float(((logits.sigmoid() > 0.5).numpy() == yte).mean())
    hist.append((tr_loss / n, te_loss, acc))
    print(f"epoch {epoch+1}  train {hist[-1][0]:.3f}  test {te_loss:.3f}  acc {acc:.3f}")

hist = np.array(hist)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(hist[:, 0], label="train loss")
ax.plot(hist[:, 1], label="test loss")
ax.set_title("Toy Transformer on feedback text")
ax.legend(); plt.tight_layout(); plt.show()
print("majority acc", 1 - yte.mean())


## Encoder vs decoder vs the API

| Shape | What it does | You have used it as |
|---|---|---|
| **Encoder** (this week) | Read a whole sequence, emit a representation | BERT, embedding models, Week 4 RAG later |
| **Decoder** | Generate the next token, one at a time, looking left | GPT, chat models |
| **Encoder–decoder** | Read a source, write a target | translation, summarization |

The four-line training step is unchanged. The `forward` is “stack of attention + feed-forward + residual skip,” repeated.

<div class="cue-box">
<strong>Visual cue — residual skip</strong>
<p><code>x = x + attention(x)</code>. Same idea as a git commit on top of the previous tree: keep the old signal, add a delta. That is why 96-layer models can still train.</p>
</div>

<div class="watch-box">
<strong>Watch out</strong>
<p>This notebook is a teaching Transformer, not a product. Do not scrape a tiny encoder and call it “we built GPT.” Production language models are pretrained on a planet of text. Your job is usually: pick a model, prompt it, fine-tune lightly, or embed + retrieve (the LangChain course).</p>
</div>

<div class="ship-box">
<strong>Ship / don’t ship</strong>
<ul>
<li><strong>Tabular churn</strong> → GBT (Week 10–12).</li>
<li><strong>Screenshots / dense grids</strong> → CNN (Week 13).</li>
<li><strong>Short sensor traces on-device</strong> → GRU maybe (Week 14).</li>
<li><strong>Language, code, mixed documents</strong> → Transformer, usually via an API or a small open model — not from-scratch on 10k comments.</li>
</ul>
</div>


## ✍️ Exercises

**1. Remove positions.** Comment out `+ self.pos(pos)`. What happens to accuracy? That is the “order is invisible” lesson.

**2. Attention map.** After training, run one sentence through `self.enc.layers[0].self_attn` (or print `weights` from the toy block). Which characters attend to the `!`?

**3. Architecture memo.** Four sentences to your VP: CNN vs RNN vs Transformer vs GBT, with one CloudWave example each.

## 🤔 Reflection

1. Attention is a join. What are the two tables?
2. Why can a Transformer use a GPU better than an RNN?
3. After this week, what is left that is *not* “just attention”? (tokenization, alignment, eval, product)

## 🎓 You now have the three pillars

| Pillar | Where |
|---|---|
| **Strong Python + NumPy + Pandas + PyTorch** | Weeks 0–2, 11 |
| **ML fundamentals** (regression, classification, overfit, bias, variance) | Weeks 6–10 |
| **Deep learning** (nets, CNN, RNN, Transformer, the training loop) | Weeks 11, 13–15 |

The LangChain course is what you do when the Transformer *already exists* and you need to wire it into a product. You are ready for it.
